In [1]:
import json

knowledge = [
    {
        "condition": "low_tenure",
        "strategy": "Implement strong onboarding and early engagement programs",
        "source": "Customer Onboarding Best Practices – HubSpot"
    },
    {
        "condition": "low_tenure",
        "strategy": "Deliver value early to prevent early churn",
        "source": "The Value of Keeping the Right Customers – HBR"
    },
    {
        "condition": "high_charges",
        "strategy": "Offer flexible pricing or discounts",
        "source": "Managing Churn to Maximize Profits – HBS"
    },
    {
        "condition": "high_charges",
        "strategy": "Optimize perceived value and pricing",
        "source": "Breaking the Back of Customer Churn – Bain"
    },
    {
        "condition": "general",
        "strategy": "Use analytics-driven retention strategies",
        "source": "Churn Prediction in Telecom – MDPI"
    }
]

with open("retention_knowledge.json", "w") as f:
    json.dump(knowledge, f, indent=4)

print("✅ JSON created successfully")

✅ JSON created successfully


In [2]:
import json

with open("retention_knowledge.json") as f:
    knowledge = json.load(f)

print(knowledge)

[{'condition': 'low_tenure', 'strategy': 'Implement strong onboarding and early engagement programs', 'source': 'Customer Onboarding Best Practices – HubSpot'}, {'condition': 'low_tenure', 'strategy': 'Deliver value early to prevent early churn', 'source': 'The Value of Keeping the Right Customers – HBR'}, {'condition': 'high_charges', 'strategy': 'Offer flexible pricing or discounts', 'source': 'Managing Churn to Maximize Profits – HBS'}, {'condition': 'high_charges', 'strategy': 'Optimize perceived value and pricing', 'source': 'Breaking the Back of Customer Churn – Bain'}, {'condition': 'general', 'strategy': 'Use analytics-driven retention strategies', 'source': 'Churn Prediction in Telecom – MDPI'}]


In [3]:
!pip install langgraph groq

from langgraph.graph import StateGraph
from typing import TypedDict, List
from groq import Groq

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [5]:
class AgentState(TypedDict):
    churn_prob: float
    tenure: int
    monthly: float

    risk_level: str
    reasons: List[str]

    strategies: List[str]
    sources: List[str]

    final_output: str

In [6]:
def risk_node(state: AgentState):

    prob = state["churn_prob"]

    if prob > 0.7:
        risk = "High"
    elif prob > 0.4:
        risk = "Medium"
    else:
        risk = "Low"

    reasons = []

    if state["tenure"] < 6:
        reasons.append("low_tenure")

    if state["monthly"] > 80:
        reasons.append("high_charges")

    if not reasons:
        reasons.append("general")

    return {
        **state,
        "risk_level": risk,
        "reasons": reasons
    }

In [7]:
def retrieval_node(state: AgentState):

    reasons = state["reasons"]

    strategies = []
    sources = []

    for item in knowledge:
        if item["condition"] in reasons:
            strategies.append(item["strategy"])
            sources.append(item["source"])

    return {
        **state,
        "strategies": list(set(strategies)),
        "sources": list(set(sources))
    }

In [8]:
def planning_node(state: AgentState):

    prompt = f"""
    You are an AI Customer Retention Strategist.

    Customer churn probability: {state['churn_prob']}
    Risk level: {state['risk_level']}
    Reasons: {state['reasons']}

    Retrieved Strategies: {state['strategies']}
    Sources: {state['sources']}

    IMPORTANT:
    Use ONLY provided strategies and sources.

    OUTPUT FORMAT:

    Risk Summary:
    ...

    Recommendations:
    1.
    2.
    3.

    Sources:
    ...

    Disclaimer:
    This prediction is probabilistic.
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )

    return {
        **state,
        "final_output": response.choices[0].message.content
    }

In [9]:
builder = StateGraph(AgentState)

builder.add_node("risk", risk_node)
builder.add_node("retrieval", retrieval_node)
builder.add_node("planning", planning_node)

builder.set_entry_point("risk")

builder.add_edge("risk", "retrieval")
builder.add_edge("retrieval", "planning")

graph = builder.compile()

In [10]:
state = {
    "churn_prob": 0.82,
    "tenure": 3,
    "monthly": 95
}

result = graph.invoke(state)

print(result["final_output"])

Risk Summary:
The customer is at high risk of churn with a probability of 0.82, primarily due to low tenure and high charges. This suggests that the customer has not yet formed a strong connection with the product or service, and the perceived value may not be aligned with the costs.

Recommendations:
1. Deliver value early to prevent early churn: To mitigate the risk of churn due to low tenure, it's essential to provide the customer with tangible value as soon as possible, demonstrating the product's or service's benefits and building a strong foundation for the relationship.
2. Optimize perceived value and pricing: Given the high charges, it's crucial to ensure that the customer perceives the value received as being commensurate with the costs incurred. This may involve re-evaluating pricing strategies or offering flexible pricing options to better align with the customer's needs and expectations.
3. Implement strong onboarding and early engagement programs: A structured onboarding p